In [ ]:
""" Importing libraries """

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Dense, Activation, MaxPooling2D, Dropout, Flatten
from keras.datasets import cifar10
from keras.utils import to_categorical
from keras.optimizers import Adam
from keras.models import load_model

import os
import pickle
import numpy as np
from tensorflow.keras.utils import to_categorical
import kagglehub

In [ ]:
""" Loading the dataset and scaling the pixels of the images for noise reduction """

path = kagglehub.dataset_download("fedesoriano/cifar100")

# Load CIFAR-100 files
def load_data(filename):
    with open(filename, 'rb') as f:
        data = pickle.load(f, encoding='bytes')

    X = data[b'data'].reshape(-1, 3, 32, 32)
    X = X.transpose(0, 2, 3, 1)
    y = np.array(data[b'fine_labels'])

    return X, y

# Load existing training and testing splits
X_train, y_train = load_data(os.path.join(path, 'train'))
X_test, y_test = load_data(os.path.join(path, 'test'))

# Normalize pixels to [0, 1]
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# One-hot encode labels (100 classes)
y_train = to_categorical(y_train, 100)
y_test = to_categorical(y_test, 100)

print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

**Why is it important to normalize the pixels of the images?**

The original image pixels have values from 0 to 255. Dividing each value by 255 converts them to a range between 0 and 1. This gives the neural network smaller, consistent input values, which helps training become more stable and often converge faster. It also prevents large pixel values from dominating the calculations. The same normalization must be applied to new images before making predictions.

In [ ]:
# Printing the shape of the training set
X_train.shape

**Discuss**

- What does this (50000, 32, 32, 3) shape mean?

**50000:** There are 50,000 individual images in the training dataset.

**32, 32:** Each image has a resolution of 32 pixels in height and 32 pixels in width.

**3:** Each image has 3 color channels corresponding to Red, Green, and Blue (RGB).


In [ ]:
""" Visualizing the images in our dataset """
# Function that displays cifar images
def display_cifar(images, size):
    n = len(images)
    plt.figure()
    plt.gca().set_axis_off()
    im = np.vstack([np.hstack([images[np.random.choice(n)] for i in range(size)]) for i in range(size)])
    plt.imshow(im)
    plt.show()

# Displaying some pictures
display_cifar(X_train, 5)

In [ ]:
""" In this section we are going to apply convolutional neural networks in detecting objects and classifying their names """
# Building a convolutional neural network without tuning hyperparameters
model = Sequential()
model.add(Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same', input_shape=(32, 32, 3)))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'))
model.add(MaxPooling2D((2, 2)))
model.add(Flatten())
model.add(Dense(128, activation='relu', kernel_initializer='he_uniform'))
model.add(Dense(100, activation='softmax'))
# compile model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=1, batch_size=100, validation_data=(X_test, y_test))

**What is the importance of Max Pooling?**

Max Pooling reduces the width and height of feature maps by selecting the maximum value from each small region (in this notebook, a 2 × 2 region). This reduces the amount of computation and the number of values passed to later layers. It also keeps the strongest detected features and makes the model less sensitive to small changes in an object's exact position.

**What is categorical cross-entropy loss?**

Categorical cross-entropy is a loss function used for multi-class classification when the correct labels are one-hot encoded. It compares the model's predicted probability for each class with the true class label. The loss becomes smaller when the model assigns a high probability to the correct class, and larger when it is confident in an incorrect class. The softmax output layer produces the class probabilities used by this loss function.

In [ ]:
# Saving the model
model.save('model.h5')

# Reloading the saved model
reloaded_model = tf.keras.models.load_model('model.h5')

In [ ]:
''' Receiving user input and classifying the image '''

from PIL import Image

# CIFAR-100 fine class names, in label order
class_names = [
    'apple', 'aquarium_fish', 'baby', 'bear', 'beaver',
    'bed', 'bee', 'beetle', 'bicycle', 'bottle',
    'bowl', 'boy', 'bridge', 'bus', 'butterfly',
    'camel', 'can', 'castle', 'caterpillar', 'cattle',
    'chair', 'chimpanzee', 'clock', 'cloud', 'cockroach',
    'couch', 'crab', 'crocodile', 'cup', 'dinosaur',
    'dolphin', 'elephant', 'flatfish', 'forest', 'fox',
    'girl', 'hamster', 'house', 'kangaroo', 'keyboard',
    'lamp', 'lawn_mower', 'leopard', 'lion', 'lizard',
    'lobster', 'man', 'maple_tree', 'motorcycle', 'mountain',
    'mouse', 'mushroom', 'oak_tree', 'orange', 'orchid',
    'otter', 'palm_tree', 'pear', 'pickup_truck', 'pine_tree',
    'plain', 'plate', 'poppy', 'porcupine', 'possum',
    'rabbit', 'raccoon', 'ray', 'road', 'rocket',
    'rose', 'sea', 'seal', 'shark', 'shrew',
    'skunk', 'skyscraper', 'snail', 'snake', 'spider',
    'squirrel', 'streetcar', 'sunflower', 'sweet_pepper', 'table',
    'tank', 'telephone', 'television', 'tiger', 'tractor',
    'train', 'trout', 'tulip', 'turtle', 'wardrobe',
    'whale', 'willow_tree', 'wolf', 'woman', 'worm'
]

# Load an image, prepare it in the same way as the training images,
# and use the saved model to predict its CIFAR-100 class.
def user_image(image_path):
    # Open the image, convert it to RGB, and resize it to 32 x 32 pixels.
    image = Image.open(image_path).convert("RGB")
    image = image.resize((32, 32))

    # Convert the image to a NumPy array and normalize pixel values to [0, 1].
    image_array = np.asarray(image, dtype=np.float32) / 255.0

    # Add a batch dimension: (32, 32, 3) becomes (1, 32, 32, 3).
    image_array = np.expand_dims(image_array, axis=0)

    # Predict probabilities for the 100 CIFAR-100 classes.
    prediction = reloaded_model.predict(image_array, verbose=0)

    # Find the class with the highest predicted probability.
    obj = int(np.argmax(prediction[0]))
    confidence = float(prediction[0][obj])

    # Display and return the predicted class and its confidence.
    print(f"The object is predicted as: {class_names[obj]}")
    print(f"Prediction confidence: {confidence:.2%}")

    return class_names[obj], confidence

In [ ]:
# Upload an image whose subject belongs to one of the CIFAR-100 classes.
# Replace the path below with the location of your uploaded image.
# Example: user_image("/content/apple.jpg")

user_image("/content/apple.jpg")

### Notes on testing the image classifier

The uploaded image should show an object or scene represented by one of the 100 CIFAR-100 fine classes. The model was trained on small 32 × 32 images, so predictions on ordinary photographs may be less reliable. A prediction confidence is the probability assigned by the model to its selected class; it is not a guarantee that the prediction is correct.